In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import time
import pandas as pd

In [2]:
def parse_quote_list(driver):
    quotes = []
    quotes_divs = driver.find_elements(By.CLASS_NAME, 'quote')
    assert len(quotes_divs) > 0, "No quotes found on the page."
    
    for quote in quotes_divs:
        text = quote.find_element(By.CLASS_NAME, 'text').text
        author = quote.find_element(By.CLASS_NAME, 'author').text
        quote_tags = quote.find_elements(By.CLASS_NAME, 'tag')
        tags = [tag.text for tag in quote_tags]
        quotes.append({
            'text': text,
            'author': author,
            'tags': tags
        })
    return quotes

def scroll_to_bottom(driver):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1)

A dynamic JS website, quotes are loaded in batches by scrolling to the bottom: https://quotes.toscrape.com/scroll

In [5]:
MAIN_URL = 'https://quotes.toscrape.com/scroll'

driver = webdriver.Chrome()
driver.get(MAIN_URL)
time.sleep(2)

for i in range(20):
    scroll_to_bottom(driver)

quotes = parse_quote_list(driver)
print(f"Scraped {len(quotes)} quotes.")             

driver.quit()

Scraped 100 quotes:


In [4]:
MAIN_URL = 'https://quotes.toscrape.com/scroll'

driver = webdriver.Chrome()
driver.get(MAIN_URL)
time.sleep(2)

while True:
    prev_num_quotes = len(driver.find_elements(By.CLASS_NAME, 'quote'))
    scroll_to_bottom(driver)
    new_num_quotes = len(driver.find_elements(By.CLASS_NAME, 'quote'))
    if new_num_quotes == prev_num_quotes:
        break
        
quotes = parse_quote_list(driver)
print(f"Scraped {len(quotes)} quotes.")

driver.quit()

Scraped 100 quotes.


In [9]:
df = pd.DataFrame(quotes)
df['tags'] = df['tags'].apply(lambda x: ', '.join(x))
df.head()

,text,author,tags
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational"


In [10]:
df.to_csv('data/quotes_scroll.csv', index=False)

A dynamic Javascript (JS) website, 10 quotes per page, 10 pages total :https://quotes.toscrape.com/js

In [5]:
MAIN_URL = 'https://quotes.toscrape.com/js'

driver = webdriver.Chrome()
driver.get(MAIN_URL)
time.sleep(2)

all_quotes = []
while True:
    try:
        page_quotes = parse_quote_list(driver)
        all_quotes.extend(page_quotes)
        next_btn = driver.find_element(By.CLASS_NAME, 'next').find_element(By.TAG_NAME, 'a')
        next_btn.click()
        time.sleep(1)
        print(f"Scraping page {len(all_quotes) // 10 + 1} with URL: {driver.current_url}")
        # or get url with next_btn.get_attribute('href')
    except NoSuchElementException:
        break

driver.quit()
print(f"Scraped {len(all_quotes)} quotes.")

Scraping page 2 with URL: https://quotes.toscrape.com/js/page/2/
Scraping page 3 with URL: https://quotes.toscrape.com/js/page/3/
Scraping page 4 with URL: https://quotes.toscrape.com/js/page/4/
Scraping page 5 with URL: https://quotes.toscrape.com/js/page/5/
Scraping page 6 with URL: https://quotes.toscrape.com/js/page/6/
Scraping page 7 with URL: https://quotes.toscrape.com/js/page/7/
Scraping page 8 with URL: https://quotes.toscrape.com/js/page/8/
Scraping page 9 with URL: https://quotes.toscrape.com/js/page/9/
Scraping page 10 with URL: https://quotes.toscrape.com/js/page/10/
Scraped 100 quotes.


In [6]:
df2 = pd.DataFrame(all_quotes)
df2['tags'] = df2['tags'].apply(lambda x: ', '.join(x))
df2.head()

,text,author,tags
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational"


In [14]:
df2.to_csv('data/quotes_pagination.csv', index=False)